In [1]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss


2026-02-14 17:28:15.000790: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771090095.238608      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771090095.310656      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771090095.877154      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771090095.877204      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771090095.877207      55 computation_placer.cc:177] computation placer alr

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("Model loaded successfully!")


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


Chat Function

In [5]:
def chat(prompt, max_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [6]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Overfitting occurs when a model performs well on training data but poorly on unseen data.",
    "Regularization helps reduce overfitting by penalizing large weights.",
    "Lasso regression uses L1 regularization.",
    "Ridge regression uses L2 regularization."
]


Create Embeddings and FAISS Index

In [7]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedder.encode(documents)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("FAISS index created!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created!


Retrieve Relevant Context

In [8]:
def retrieve_context(query, k=2):
    query_embedding = embedder.encode([query])
    _, indices = index.search(query_embedding, k)
    return " ".join([documents[i] for i in indices[0]])

RAG-Enhanced Chat

In [9]:
def rag_chat(query):
    context = retrieve_context(query)
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    return chat(prompt)


test llm

In [10]:
query = "What is overfitting in machine learning?"
response = rag_chat(query)

print("Question:", query)
print("Answer:", response)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is overfitting in machine learning?
Answer: (4).


In [12]:
print("RAG Chatbot (type 'exit' to stop)")
while True:
    q = input("You: ")
    if q.lower() == "exit":
        break
    print("Bot:", rag_chat(q))


RAG Chatbot (type 'exit' to stop)


You:  machine learning


Bot: What is a subset of artificial intelligence?


You:  overfillting


Bot: is penalizing large weights


You:  exit
